# 01 — Coleta Atlas (GET) — documentação da disciplina

**Projeto:** Preditor de Falhas ML (Grupo 16)
**Disciplinas:** ED2 + Redes + APS
**Card:** S1.2 — GET `/results/` (camada raw)

> Este notebook é a **documentação / evidência** pedida pela professora.
> **Não** é o pipeline de produção.
> O **core** do GET fica no pacote Python: `fetch_measurement_results`
> (`src/preditor_de_falhas_ml/atlas.py`). A CLI `getResults` e a Lambda (S1.7)
> reutilizam a mesma função — não este notebook.
> HTTP e autenticação (`requests.request` + `Authorization`) vivem em `_request`.
> **Não** criamos medição aqui (sem POST). Só lemos `/results/` de um `msm_id` já existente.

**Objetivo:** consumir resultados de uma medição do RIPE Atlas, receber o JSON,
exibi-lo em um `DataFrame` e gravar a camada `raw` em `data/raw/` (equivalente
à pasta `raw` do template da disciplina; não usamos Google Drive).

> Esta etapa cobre **somente a ingestão e o armazenamento da camada `raw`**.
> Não agregue janelas, não crie variáveis, não rotule `status_real` e não treine
> modelos aqui.

## Entrega mínima

- Justificar as decisões da equipe sobre os parâmetros da consulta;
- Executar uma requisição válida à API via `fetch_measurement_results` (HTTP no pacote);
- Exibir os dados em um `DataFrame`;
- Salvar o JSON original da resposta;
- Salvar os registros em **CSV** na pasta `data/raw/`;
- Registrar evidências da coleta: URL, parâmetros, horário, quantidade de registros e nomes dos arquivos.


## 1. Identificação da equipe

| Integrante | Papel nesta entrega |
|---|---|
| Guilherme Leite Tavares | Arquitetura de dados, Quadro Kanban, especificação e revisão |
| Alexandre Tiago de Oliveira | N/A |
| Ingrid Ferreira de Sousa | N/A |
| Kauan Garcia Dias de Oliveira | Revisão do notebook / aprovação do PR |
| Lucas Eduardo Malachias Bagatela | N/A |
| Stephanie Vitoria Bessa dos Santos | Revisão / aprovação de PRs |

- **Turma:** Ciência da Computação 3 semestre e 4 semestre
- **Data da coleta (demo no notebook):** 2026-09-14
- **Repo:** https://github.com/g-tavares14/preditor-de-falhas-ml


## 2. Decisões da equipe

| Decisão | Escolha | Por quê |
|---|---|---|
| API | RIPE Atlas | Fonte definida do projeto; não usamos outra API |
| Verbo nesta fase | **GET** `/measurements/{msm_id}/results/` | Coleta/ingestão; POST (criar medição) foi setup (S1.6/S1.8), não esta demo |
| Tipo da medição | ping | Tipo principal do dataset do preditor; traceroute entra no POST (notebook 02) |
| `msm_id` da demo | `210928969` (ping → 94.140.14.14 AdGuard) | Um dos 8 IDs reais do hub; série periódica já criada |
| Janela | 2026-09-13 12:00 UTC → 2026-09-14 12:00 UTC | Período em que a série S1.8 gerou histórico (hoje a série está Stopped; GET do histórico permanece) |
| Onde está o código | Pacote `src/preditor_de_falhas_ml/atlas.py` | CLI e Lambda reutilizam a mesma função; células **não** chamam `requests` |
| Papel deste notebook | Documentação + evidência da disciplina | Não é o pipeline AWS |
| Formato tabular | **CSV** | Exigência da fase; fácil de abrir na correção |
| Pasta | `data/raw/` | Camada raw do repo (equivale à pasta `raw` pedida no template) |


## 3. Bibliotecas

Na raiz do repositório: `uv sync` (não usamos `pip` nem o template Colab).
`requests` **não** entra nas células — só dentro de `_request` em `atlas.py`.
Defina `RIPE_ATLAS_API_KEY` no ambiente ou `.env` (**não** commitado).


In [ ]:
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from preditor_de_falhas_ml import append_data, fetch_measurement_results

print("Bibliotecas carregadas.")
print("HTTP: fetch_measurement_results -> atlas._request (não requests nas células).")

## 4. Preparar a pasta `raw`

O template da disciplina usa Google Drive (`ripe_atlas/raw`). Aqui a camada
equivalente é `data/raw/` no repositório. A subpasta **precisa** chamar-se `raw`.
Os arquivos gerados estão no `.gitignore` (exceto `.gitkeep`).


In [ ]:
PASTA_RAW = Path("data/raw")
PASTA_RAW.mkdir(parents=True, exist_ok=True)
print("Arquivos raw serão salvos em:", PASTA_RAW.resolve())

## 5. Definir os parâmetros da consulta

Os mesmos da função do pacote e da CLI `getResults`:

- `msm_id` — id de uma medição Atlas já existente (sem POST nesta demo);
- `start` / `stop` — janela Unix UTC;
- `FORMATO` — tabular da disciplina (`csv`);
- `GRAVAR_RAW` — nesta entrega fica `True` (evidência obrigatória).

Na raiz: `uv sync`. A chave `RIPE_ATLAS_API_KEY` vem do ambiente (ou `.env`).


In [ ]:
msm_id = 210928969
start = int(datetime(2026, 9, 13, 12, 0, tzinfo=UTC).timestamp())
stop = int(datetime(2026, 9, 14, 12, 0, tzinfo=UTC).timestamp())
GRAVAR_RAW = True
FORMATO = "csv"

URL = f"https://atlas.ripe.net/api/v2/measurements/{msm_id}/results/"
PARAMETROS = {"start": start, "stop": stop}

api_key = os.environ.get("RIPE_ATLAS_API_KEY", "")

assert isinstance(msm_id, int) and msm_id > 0
assert stop > start
assert FORMATO == "csv"
assert GRAVAR_RAW

print("URL:", URL)
print("Início (UTC):", datetime.fromtimestamp(start, tz=UTC).isoformat())
print("Fim (UTC):", datetime.fromtimestamp(stop, tz=UTC).isoformat())
print("Parâmetros:", PARAMETROS)
print("GRAVAR_RAW:", GRAVAR_RAW, "FORMATO:", FORMATO)
print("Chave definida:", bool(api_key))

## 6. Consumir a API e obter o JSON

A célula abaixo **chama** `fetch_measurement_results` (importada na seção 3).
Não usa `requests` nas células e **não** chama `get_data` (POST one-off).

O HTTP (status, timeout, `Authorization: Key …`) fica em `atlas._request`.
Sucesso nesta demo = DataFrame com pelo menos um registro (equivalente ao
“JSON não vazio / HTTP 200” do template). Se vier vazio, revise `msm_id` ou a janela.


In [ ]:
if not api_key:
    raise RuntimeError(
        "RIPE_ATLAS_API_KEY não está definida (ambiente ou .env, não commitado)."
    )

instante_requisicao_utc = datetime.now(tz=UTC)
df_raw = fetch_measurement_results(
    api_key,
    msm_id,
    start=start,
    stop=stop,
)

if df_raw.empty:
    raise ValueError("A consulta não retornou registros. Reveja o msm_id ou o período.")

dados_json = json.loads(df_raw.to_json(orient="records", force_ascii=False))

print("Função:", "fetch_measurement_results")
print("Quantidade de registros recebidos:", len(dados_json))
print("Tipo do objeto principal:", type(dados_json).__name__)
print("Coletado em (UTC):", instante_requisicao_utc.isoformat())

## 7. Inspecionar uma pequena amostra do JSON

Exiba somente o primeiro registro para compreender a estrutura sem poluir o
notebook. A resposta completa será preservada em arquivo. O dicionário vem do
DataFrame bruto (`pd.DataFrame(results)` dentro do pacote); campos aninhados
como `result[]` do ping continuam como objetos.


In [ ]:
print(json.dumps(dados_json[0], indent=2, ensure_ascii=False, default=str)[:5000])

## 8. Transformar o JSON em `DataFrame`

O template da disciplina usa `pd.json_normalize`. Aqui o GET do pacote já
devolve `pd.DataFrame(results)` — o mesmo objeto que a CLI `getResults` e a
Lambda usam. Não removemos nem agregamos registros.


In [ ]:
print("Dimensões do DataFrame (linhas, colunas):", df_raw.shape)
display(df_raw.head())

In [ ]:
# Conferência básica das colunas e tipos identificados.
df_raw.info()

## 9. Salvar os dados na pasta `raw`

Serão gerados:

1. Um arquivo `.json` com os registros da resposta (lista de objetos do GET);
2. Um arquivo `.csv` com o conteúdo exibido no `DataFrame`;
3. Um arquivo de metadados `.json` com as evidências da coleta;
4. (Extra do repo) append em `measurements.jsonl` via `append_data` — contrato
   local do preditor, além da tríade da disciplina.

O timestamp UTC no nome evita sobrescrever coletas anteriores.
Produção grava no S3 (S1.7); isto é só evidência da disciplina.


In [ ]:
assert GRAVAR_RAW, "Gravar dados brutos é obrigatório nesta entrega."
assert len(df_raw) > 0, "Não há dados para gravar."
assert FORMATO == "csv"

stamp = instante_requisicao_utc.strftime("%Y%m%dT%H%M%SZ")
nome_base = f"ripe_atlas_m{msm_id}_{stamp}"

caminho_json = PASTA_RAW / f"{nome_base}.json"
caminho_tabela = PASTA_RAW / f"{nome_base}.csv"
caminho_metadados = PASTA_RAW / f"{nome_base}_metadata.json"

caminho_json.write_text(
    json.dumps(dados_json, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
df_raw.to_csv(caminho_tabela, index=False, encoding="utf-8")

metadados = {
    "measurement_id": msm_id,
    "url": URL,
    "parametros": PARAMETROS,
    "inicio_utc": datetime.fromtimestamp(start, tz=UTC).isoformat(),
    "fim_utc": datetime.fromtimestamp(stop, tz=UTC).isoformat(),
    "coletado_em_utc": instante_requisicao_utc.isoformat(),
    "funcao": "fetch_measurement_results",
    "http": "atlas._request (GET /api/v2/measurements/{msm_id}/results/)",
    "quantidade_registros": int(len(df_raw)),
    "quantidade_colunas_dataframe": int(len(df_raw.columns)),
    "arquivo_json": caminho_json.name,
    "arquivo_tabelar": caminho_tabela.name,
    "nota": "Demo notebook 01; core = fetch_measurement_results",
}

caminho_metadados.write_text(
    json.dumps(metadados, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

caminho_jsonl = append_data(df_raw, output_dir=PASTA_RAW)

{
    "json": str(caminho_json),
    "csv": str(caminho_tabela),
    "metadata": str(caminho_metadados),
    "jsonl": str(caminho_jsonl),
}

## 10. Verificar se a entrega foi salva

A célula abaixo verifica a existência dos arquivos da tríade da disciplina
(JSON + CSV + metadata) e lista tamanhos. O JSONL é extra do contrato do repo.


In [ ]:
assert caminho_json.exists(), f"JSON ausente: {caminho_json}"
assert caminho_tabela.exists(), f"CSV ausente: {caminho_tabela}"
assert caminho_metadados.exists(), f"Metadados ausentes: {caminho_metadados}"
assert caminho_jsonl.exists(), f"JSONL ausente: {caminho_jsonl}"

arquivos_gerados = pd.DataFrame(
    {
        "arquivo": [
            caminho_json.name,
            caminho_tabela.name,
            caminho_metadados.name,
            caminho_jsonl.name,
        ],
        "tamanho_bytes": [
            caminho_json.stat().st_size,
            caminho_tabela.stat().st_size,
            caminho_metadados.stat().st_size,
            caminho_jsonl.stat().st_size,
        ],
    }
)
display(arquivos_gerados)
print("Coleta concluída e salva na pasta raw.")

**Observação**

Evidências aceitas:

1. histórico de edição de documento compartilhado;
2. commit ou pull request no GitHub;
3. arquivo ou trecho de código produzido;
4. relatório, planilha, diagrama, apresentação ou rascunho;
5. registro de tarefa no Trello, GitHub Projects ou ferramenta semelhante;
6. registro de testes realizados;
7. print de reunião, conversa ou e-mail relacionado à atividade;
8. outro material que demonstre claramente a contribuição individual.

A evidência deverá identificar o aluno e estar relacionada à atividade declarada.
Sempre que possível, apresente materiais com data, autoria ou histórico de edição.
Verifique se todos os links estão acessíveis. Trabalhos realizados em conjunto
também devem indicar a contribuição específica de cada integrante. Respostas
genéricas, como “ajudei no trabalho”, “fiz a pesquisa” ou “participei do código”,
não serão consideradas suficientes. Prints de conversas podem complementar a
comprovação, mas não devem ser a única evidência quando houver um produto
verificável. A ausência de descrição ou de evidência poderá impedir a validação
da contribuição individual.


## 11. Checklist da equipe

- [x] Identificamos os integrantes e a turma;
- [x] Justificamos o ID da medição, o tipo (ping) e o intervalo escolhido;
- [x] A requisição foi um GET válido via `fetch_measurement_results` (HTTP em `atlas._request`);
- [x] O JSON / DataFrame retornou pelo menos um registro;
- [x] Exibimos o `DataFrame` no notebook (`head` / `info`);
- [x] Salvamos o JSON original na pasta `data/raw/`;
- [x] Salvamos o DataFrame em CSV na pasta `data/raw/`;
- [x] Mantivemos o arquivo de metadados da coleta (nome com timestamp UTC);
- [x] Não realizamos limpeza, agregação, rotulação ou treinamento nesta etapa;
- [x] Sem `requests` / sem POST / sem `get_data` nas células;
- [x] Claro para a banca: **notebook = doc**; **produção = pacote + Lambda GET**.

### Registro final da equipe

Coletamos resultados de ping da medição `210928969` (AdGuard `94.140.14.14`, um
dos 8 `msm_id` do hub) na janela 2026-09-13 12:00 UTC → 2026-09-14 12:00 UTC,
período em que a série S1.8 gerou histórico. O GET foi o do pacote
(`fetch_measurement_results`), não um `requests` nas células. O volume obtido
aparece na célula de verificação (quantidade de registros + tamanhos dos arquivos
em `data/raw/`). CSV é o formato tabular desta fase; JSON e metadata fecham a
tríade de evidência. Sem agregação e sem `status_real` aqui.
